# 03 · Observability: OpenTelemetry GenAI → Azure Monitor + Datadog
Agents emit spans with GenAI semantic conventions (`gen_ai.operation.name`, `gen_ai.request.model`, `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens`, `gen_ai.agent.id`, `gen_ai.agent.name`) and resource attributes (`service.name`, `cloud.provider`).
The OTel Collector (`otel/collector-config.yaml`) fans out to **Datadog** and **Azure Monitor**. The APIM gateway adds `llm-emit-token-metric` (namespace `mccp`).
**Datadog alerts, the gateway enforces.**

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mccp").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

RUN_LIVE = False  # True only after deploying infra/mccp and filling .env (see docs/mccp/PLAN.md)

from mccp.config import Settings
from mccp.report import Console
settings = Settings.from_env(mode="azure" if RUN_LIVE else "local")
c = Console(color=False)
print("mode:", settings.mode, "(SIMULATED)" if settings.is_local else "(LIVE)")

In [ ]:
from mccp.demo import build_local_runtime
rt = build_local_runtime(settings)
for agent_id, user in [("contoso-claims-triage", "maria.gomez@contoso-health.example"),
                       ("contoso-patient-scheduling", "r.silva@contoso-health.example")]:
    rt.call(agent_id, user, "status check")
print(rt.telemetry.dump(last=1))
print(rt.simulator.metrics[-1])

## Live telemetry (RUN_LIVE) — needs APPLICATIONINSIGHTS_CONNECTION_STRING and/or OTEL_EXPORTER_OTLP_ENDPOINT

In [ ]:
if RUN_LIVE:
    from mccp.telemetry import setup_telemetry, genai_attributes
    tel = setup_telemetry(settings)
    agent = rt.agents[0]
    attrs = genai_attributes(agent, agent.model, 120, 40)
    with tel.span(f"chat {agent.model}", attrs, agent.cloud):
        tel.token_usage(attrs, 120, 40)
    print("exported via", tel.exporters)
else:
    print("RUN_LIVE=False: console recorder only")

## Datadog monitor & webhook
`datadog/monitor_token_budget.json` alerts per `gen_ai.agent.id`; its webhook (`datadog/webhook_payload.json`) calls the same kill-switch Logic App as Azure Monitor.

In [ ]:
import json
print(json.dumps(json.loads((ROOT / "datadog/monitor_token_budget.json").read_text())["query"], indent=2))